# Cross-Lingual Zero-Shot Hate Speech Detection on AfriHate

**Research question** (from `Project_description.txt`):
*Can a model fine-tuned only on **higher-resource** African languages (Hausa + Amharic) detect hate speech in **lower-resource** African languages (Twi, Nigerian Pidgin) **without** any target-language training data?*

**Why this matters.** Most African languages lack labeled hate-speech data. If a multilingual encoder fine-tuned on a few well-resourced languages transfers to many others, low-resource communities benefit from moderation tooling without each language needing its own labeled corpus. If transfer is uneven, that itself is a finding: the gap quantifies how much a language is *underserved* by today's multilingual NLP.

**Pipeline (this notebook).**
1. Fine-tune a multilingual encoder (configurable via `NLP_MODEL`) on a configurable set of source AfriHate languages (`NLP_SOURCE_LANGS`, default `hau,amh`).
2. **Zero-shot** evaluation on **Twi (`twi`)** and **Nigerian Pidgin (`pcm`)** test sets — no parameter updates on targets.
3. **In-distribution** test on the held-out source split — sanity ceiling, with per-source-language breakdown.
4. **Architecture comparison** with the paper's strong `Davlan/afro-xlmr-large-76L` (or vice-versa, depending on `NLP_MODEL`).
5. **Consolidated report table** for the current run **and** an aggregated `experiment_log.csv` that accumulates results across runs so size, source-language and learning-rate variations can be compared in one place.

**Experiment matrix tracked in `Checkpoints/experiment_log.csv`.**

> **Shell hygiene:** `jupyter nbconvert` inherits your shell environment. If `NLP_SOURCE_LANGS` or `NLP_EXPERIMENT_ID` were exported earlier, they override defaults. Cell 1 prints **RUN_MANIFEST** — verify before long runs. Use `unset NLP_EXPERIMENT_ID NLP_SOURCE_LANGS` for a clean slate.


| ID | Model | Source langs | Notes |
|---|---|---|---|
| `E1_base_hau_amh`     | `afro-xlmr-base`     | hau+amh     | Baseline. |
| `E2_large76L_hau_amh` | `afro-xlmr-large-76L` | hau+amh    | Capacity scaling: tests if a stronger encoder closes the cross-lingual gap on its own. |
| `E3_base_hau_amh_yor` | `afro-xlmr-base`     | hau+amh+yor | Language-coverage hypothesis: adding Niger-Congo Yoruba should help Twi (also Niger-Congo) without hurting Hausa/Amharic/Pidgin. |

Re-running with different env vars appends new rows; existing rows for the same `(experiment_id, model, source_languages)` key are overwritten.

**Out of scope here, continued in `Phase2_FewShot_And_ErrorAnalysis.ipynb`.** Few-shot adaptation curves and qualitative error analysis on the strongest source model.

**Ethics.** Cross-lingual transfer can amplify false positives on dialect / in-group slang. Any deployment requires human review (see §9).


## 0. Setup — imports, paths, authentication


In [1]:
import os
import warnings
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
import torch
from torch import nn

from datasets import (
    ClassLabel,
    Value,
    concatenate_datasets,
    load_dataset,
)
from datasets.exceptions import DatasetNotFoundError
from datasets.utils.logging import disable_progress_bar as _disable_datasets_pbar

from huggingface_hub import get_token, login, whoami
from huggingface_hub.utils import disable_progress_bars as _disable_hub_pbars

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_fscore_support,
)

from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from transformers import logging as transformers_logging

# Quiet, deterministic-looking notebook output (no widgets, no warnings, no tqdm bars)
_disable_datasets_pbar()
_disable_hub_pbars()
transformers_logging.set_verbosity_error()
warnings.filterwarnings("ignore", message=r".*pin_memory.*")


def load_dotenv(env_file: Path) -> None:
    if not env_file.is_file():
        return
    for raw in env_file.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key, val = key.strip(), val.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = val


def _project_root() -> Path:
    """FinalProject root even when the notebook lives under experiments/."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "requirements.txt").is_file() and (p / "execute_notebook.sh").is_file():
            return p
    return here


PROJECT_DIR = _project_root()
os.chdir(PROJECT_DIR)
for _candidate in (PROJECT_DIR / ".env", PROJECT_DIR.parent / ".env", PROJECT_DIR.parent.parent / ".env"):
    if _candidate.is_file():
        load_dotenv(_candidate)
        break

hf_token = os.getenv("HF_TOKEN") or get_token()
if not hf_token:
    raise RuntimeError(
        "AfriHate is gated. Create a token at https://huggingface.co/settings/tokens, "
        "accept the dataset terms at https://huggingface.co/datasets/afrihate/afrihate, "
        "then put HF_TOKEN in .env or run `hf auth login`."
    )
try:
    whoami()
except Exception:
    login(token=hf_token)

CHECKPOINT_DIR = str(PROJECT_DIR / "Checkpoints")
FINAL_MODEL_PATH = str(PROJECT_DIR / "Final_Source_Model")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_PATH, exist_ok=True)

print(f"checkpoint_dir: {CHECKPOINT_DIR}")
print(f"final_model_dir: {FINAL_MODEL_PATH}")


checkpoint_dir: /Users/macpro/Desktop/MastersICS/Natural Language Processing/Final_Project/FinalProject/Checkpoints
final_model_dir: /Users/macpro/Desktop/MastersICS/Natural Language Processing/Final_Project/FinalProject/Final_Source_Model


## 1. Data — Hausa + Amharic AfriHate splits

We use AfriHate's **`hau` (Hausa)** and **`amh` (Amharic)** configs as **source** training data. Both are among the larger, better-annotated configs in AfriHate, and together they cover both Latin (Hausa) and Ge'ez (Amharic) script families — useful breadth for cross-lingual transfer.

Labels in AfriHate are `Abuse`, `Hate`, `Normal`. We map them to integers, keep a `lang` column for per-language diagnostics, and create a **stratified 15% validation split** so each class is proportionally represented during model selection.


In [ ]:
SOURCE_LANGS = [
    cfg.strip()
    for cfg in os.environ.get("NLP_SOURCE_LANGS", "hau,amh").split(",")
    if cfg.strip()
]
_MANIFEST_MODEL = os.environ.get("NLP_MODEL", "Davlan/afro-xlmr-base").strip() or "Davlan/afro-xlmr-base"
_src_key = "+".join(SOURCE_LANGS)
_explicit_exp = os.environ.get("NLP_EXPERIMENT_ID", "").strip()
if _explicit_exp:
    EXPERIMENT_ID = _explicit_exp
else:
    mn = _MANIFEST_MODEL.lower()
    if "large-76l" in mn and _src_key == "hau+amh":
        EXPERIMENT_ID = "E2_large76L_hau_amh"
    elif "afro-xlmr-base" in mn and _src_key == "hau+amh":
        EXPERIMENT_ID = "E1_base_hau_amh"
    elif "afro-xlmr-base" in mn and _src_key == "hau+amh+yor":
        EXPERIMENT_ID = "E3_base_hau_amh_yor"
    elif "large-76l" in mn and _src_key == "hau+amh+yor":
        EXPERIMENT_ID = "E4_large76L_hau_amh_yor"
    else:
        slug = _MANIFEST_MODEL.split("/")[-1].replace("-", "_")
        EXPERIMENT_ID = f"E_custom_{slug}__{_src_key.replace('+', '_')}"

print("--- RUN_MANIFEST (unset stale exports: `unset NLP_EXPERIMENT_ID NLP_SOURCE_LANGS`) ---")
print(f"  NLP_MODEL -> {_MANIFEST_MODEL}")
print(f"  NLP_SOURCE_LANGS -> {SOURCE_LANGS}  (key={_src_key})")
print(f"  experiment_id -> {EXPERIMENT_ID}")
for _k in ("NLP_LR", "NLP_NUM_EPOCHS", "NLP_MAX_LENGTH", "NLP_MAX_TRAIN_STEPS", "NLP_USE_CLASS_WEIGHTS"):
    if os.environ.get(_k):
        print(f"  {_k}={os.environ.get(_k)}")
print("--- end manifest ---\n")

print(f"experiment_id: {EXPERIMENT_ID}")
print(f"loading AfriHate splits for source languages: {SOURCE_LANGS}")

source_per_lang_train = []
source_per_lang_test = []
for lang_code in SOURCE_LANGS:
    try:
        afrihate_lang = load_dataset("afrihate/afrihate", lang_code, token=hf_token)
    except DatasetNotFoundError as exc:
        if "gated" in str(exc).lower() or "ask for access" in str(exc).lower():
            raise RuntimeError(
                "Dataset is gated: open https://huggingface.co/datasets/afrihate/afrihate "
                "while logged into the same account as your HF token, click "
                "**Agree and access repository**, then re-run this cell."
            ) from exc
        raise
    source_per_lang_train.append(
        afrihate_lang["train"].add_column("lang", [lang_code] * len(afrihate_lang["train"]))
    )
    source_per_lang_test.append(
        afrihate_lang["test"].add_column("lang", [lang_code] * len(afrihate_lang["test"]))
    )

source_train_raw = concatenate_datasets(source_per_lang_train)
source_test_raw = concatenate_datasets(source_per_lang_test)

unique_labels = sorted(set(source_train_raw["label"]))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}
label_names = [id2label[i] for i in range(len(unique_labels))]


def encode_labels(example):
    return {"labels": label2id[example["label"]]}


source_train_raw = source_train_raw.map(encode_labels).remove_columns(["label"])
source_test_raw = source_test_raw.map(encode_labels).remove_columns(["label"])

class_label_feature = ClassLabel(num_classes=len(label_names), names=label_names)
source_train_raw = source_train_raw.cast_column("labels", class_label_feature)
source_test_raw = source_test_raw.cast_column("labels", class_label_feature)

train_dev_split = source_train_raw.train_test_split(
    test_size=0.15, seed=42, stratify_by_column="labels"
)
train_dataset = train_dev_split["train"]
dev_dataset = train_dev_split["test"]
test_dataset = source_test_raw

SOURCE_TEST_SUBSET_LABEL = _src_key + " (official test)"

label_distribution = pd.Series(train_dataset["labels"]).map(id2label).value_counts()
per_lang_train_counts = pd.Series(train_dataset["lang"]).value_counts()

print(f"label mapping: {label2id}")
print(f"train: {len(train_dataset)} | dev: {len(dev_dataset)} | test: {len(test_dataset)}")
print("\ntrain class distribution:")
print(label_distribution.to_string())
print("\ntrain per-language counts:")
print(per_lang_train_counts.to_string())


## 2. Tokenization

We use the multilingual subword tokenizer matching our base model (`Davlan/afro-xlmr-base` by default; override with `NLP_MODEL`, e.g. `Davlan/afro-xlmr-large` if VRAM allows). `MAX_LENGTH=128` is sufficient for AfriHate tweets and keeps MPS training stable; raise via `NLP_MAX_LENGTH`.


In [3]:
MODEL_NAME = os.environ.get("NLP_MODEL", "Davlan/afro-xlmr-base").strip() or "Davlan/afro-xlmr-base"
MAX_LENGTH = int(os.environ.get("NLP_MAX_LENGTH", "128"))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_batch(examples):
    if "tweet" in examples:
        texts = examples["tweet"]
    elif "text" in examples:
        texts = examples["text"]
    else:
        raise KeyError("Expected 'tweet' or 'text' column in dataset.")
    encoded = tokenizer(texts, padding="max_length", truncation=True, max_length=MAX_LENGTH)
    encoded["labels"] = [int(x) for x in examples["labels"]]
    if "lang" in examples:
        encoded["lang"] = list(examples["lang"])
    return encoded


def tokenize_dataset(dataset):
    keep = [c for c in dataset.column_names if c == "lang"]
    drop = [c for c in dataset.column_names if c not in keep]
    out = dataset.map(tokenize_batch, batched=True, remove_columns=drop)
    return out.cast_column("labels", Value("int64"))


train_tokens = tokenize_dataset(train_dataset)
dev_tokens = tokenize_dataset(dev_dataset)
test_tokens = tokenize_dataset(test_dataset)

print(f"model: {MODEL_NAME}")
print(f"max_length: {MAX_LENGTH}")
print(f"tokenized train: {len(train_tokens)} | dev: {len(dev_tokens)} | test: {len(test_tokens)}")


model: Davlan/afro-xlmr-large-76L
max_length: 128
tokenized train: 9663 | dev: 1706 | test: 2615


## 3. Evaluation metrics & class-collapse diagnostics

Plain accuracy is misleading on imbalanced data — a classifier that always predicts `Normal` would score ~0.45 here. We track:

- **Macro / weighted F1** — primary task metrics, robust to class skew (`zero_division=0`).
- **Per-class F1** — surfaces which class the model is missing.
- **Balanced accuracy** — chance level on a 3-class problem is ~0.33; values >0.6 indicate real signal.
- **Matthews correlation coefficient (MCC)** — single summary score; 0 = random, 1 = perfect.
- **`pred_majority_frac`** and **`num_pred_classes_used`** — collapse diagnostics. If the model predicts only one class, `num_pred_classes_used = 1` and `pred_majority_frac ≈ 1`, regardless of accuracy. This separates *model collapse* from *broken metrics*.

The same metric set is computed for **zero-shot** evaluation (§6) and the **in-distribution test** (§7), giving consistent reporting across settings.


In [4]:
def evaluate_predictions(labels: np.ndarray, predictions: np.ndarray) -> dict:
    "Compute the full metric panel used in every evaluation in this study."
    labels = np.asarray(labels).astype(np.int64).ravel()
    predictions = np.asarray(predictions).astype(np.int64).ravel()
    n_classes = len(unique_labels)
    class_indices = list(range(n_classes))

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", labels=class_indices, zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", labels=class_indices, zero_division=0
    )
    _, _, f1_per_class, _ = precision_recall_fscore_support(
        labels, predictions, average=None, labels=class_indices, zero_division=0
    )

    pred_counts = np.bincount(predictions, minlength=n_classes)
    pred_majority_frac = float(pred_counts.max()) / max(1, len(predictions))
    num_pred_classes_used = int((pred_counts > 0).sum())

    metrics = {
        "accuracy": float(accuracy_score(labels, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "mcc": float(matthews_corrcoef(labels, predictions)),
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "precision_weighted": float(p_weighted),
        "recall_weighted": float(r_weighted),
        "pred_majority_frac": pred_majority_frac,
        "num_pred_classes_used": float(num_pred_classes_used),
    }
    for class_idx, class_f1 in enumerate(f1_per_class):
        metrics[f"f1_{label_names[class_idx]}"] = float(class_f1)
    return metrics


def compute_metrics(eval_pred) -> dict:
    logits, labels = eval_pred
    predictions = np.argmax(logits.astype(np.float64), axis=-1)
    metrics = evaluate_predictions(labels, predictions)
    metrics["f1"] = metrics["f1_macro"]
    return metrics


## 4. Fine-tuning on Hausa + Amharic

Standard XLM-R fine-tune: AdamW, learning rate `2e-5`, linear warmup (~6%), `max_grad_norm=1.0`, 3 epochs, effective batch size 16. We deliberately **do not** use class weights or label smoothing by default — the AfriHate class skew is mild (~2×), and combining weighted CE with aggressive clipping was the cause of an earlier NaN-loss / single-class collapse on MPS.

**Best checkpoint** is selected by `f1_macro` (not accuracy or balanced accuracy), so a degenerate model that predicts a single class can never tie its way to "best".

Knobs (env vars): `NLP_MODEL`, `NLP_SOURCE_LANGS` (e.g. `hau,amh,yor`), `NLP_EXPERIMENT_ID`, `NLP_NUM_EPOCHS`, `NLP_LR`, `NLP_MAX_LENGTH`, `NLP_USE_CLASS_WEIGHTS=1` for balanced (capped) class weights, `NLP_LABEL_SMOOTHING`, `NLP_MAX_TRAIN_STEPS` for smoke runs.


In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

is_cuda = torch.cuda.is_available()
is_mps = torch.backends.mps.is_available()

if is_cuda:
    print(f"device: CUDA ({torch.cuda.get_device_name(0)})")
elif is_mps:
    print("device: Apple MPS")
else:
    print("device: CPU (training will be slow)")


def get_training_profile() -> dict:
    large = "large" in MODEL_NAME.lower()
    if is_cuda:
        if large:
            return {"per_device_train_batch_size": 4, "gradient_accumulation_steps": 4}
        return {"per_device_train_batch_size": 8, "gradient_accumulation_steps": 2}
    if is_mps:
        if large:
            return {"per_device_train_batch_size": 2, "gradient_accumulation_steps": 8}
        return {"per_device_train_batch_size": 8, "gradient_accumulation_steps": 2}
    if large:
        return {"per_device_train_batch_size": 2, "gradient_accumulation_steps": 8}
    return {"per_device_train_batch_size": 4, "gradient_accumulation_steps": 4}


num_epochs = int(os.environ.get("NLP_NUM_EPOCHS", "3"))
if os.environ.get("NLP_LR", "").strip():
    learning_rate = float(os.environ["NLP_LR"])
else:
    mn = MODEL_NAME.lower()
    if "large-76l" in mn or ("xlmr-large" in mn and "base" not in mn):
        learning_rate = float(os.environ.get("NLP_LR_LARGE_DEFAULT", "1.5e-5"))
    else:
        learning_rate = 2e-5
use_class_weights = os.environ.get("NLP_USE_CLASS_WEIGHTS", "0") == "1"
label_smoothing = float(os.environ.get("NLP_LABEL_SMOOTHING", "0.0"))
smoke_steps = int(os.environ.get("NLP_MAX_TRAIN_STEPS", "-1"))

profile = get_training_profile()
effective_batch = profile["per_device_train_batch_size"] * profile["gradient_accumulation_steps"]
optimizer_steps_per_epoch = max(1, (len(train_tokens) + effective_batch - 1) // effective_batch)
total_optimizer_steps = smoke_steps if smoke_steps > 0 else optimizer_steps_per_epoch * num_epochs
warmup_steps = max(20, int(0.06 * total_optimizer_steps))

eval_steps = save_steps = max(20, total_optimizer_steps // 6)
log_steps = max(10, total_optimizer_steps // 12)

class_weights_tensor: torch.Tensor | None = None
if use_class_weights:
    from sklearn.utils.class_weight import compute_class_weight

    class_weights_array = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(len(unique_labels)),
        y=np.asarray(train_dataset["labels"], dtype=np.int64),
    )
    cap = float(os.environ.get("NLP_CLASS_WEIGHT_CAP", "3"))
    class_weights_array = np.minimum(class_weights_array, np.min(class_weights_array) * cap)
    class_weights_tensor = torch.tensor(class_weights_array, dtype=torch.float32)
    print("class_weights:", dict(zip(label_names, class_weights_array.tolist())))

print(f"model: {MODEL_NAME}")
print(f"epochs: {num_epochs} | effective_batch: {effective_batch} | lr: {learning_rate}")
print(f"label_smoothing: {label_smoothing} | use_class_weights: {use_class_weights}")
print(f"warmup_steps: {warmup_steps} | total_optimizer_steps: {total_optimizer_steps}")


class StripLangCollator:
    "Keep `lang` in the dataset for analysis but strip it before feeding the model."

    def __init__(self, tokenizer):
        self._inner = DataCollatorWithPadding(tokenizer)

    def __call__(self, features):
        return self._inner([{k: v for k, v in f.items() if k != "lang"} for f in features])


class WeightedTrainer(Trainer):
    "Trainer with optional class-weighted CE + label smoothing."

    def __init__(self, *args, class_weights=None, label_smoothing=0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self._class_weights = class_weights
        self._label_smoothing = label_smoothing

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = (
            self._class_weights.to(device=logits.device, dtype=logits.dtype)
            if self._class_weights is not None
            else None
        )
        loss_fct = nn.CrossEntropyLoss(weight=weight, label_smoothing=self._label_smoothing)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


model_init_kwargs = dict(num_labels=len(unique_labels), label2id=label2id, id2label=id2label)
if is_mps:
    model_init_kwargs["attn_implementation"] = "eager"

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, **model_init_kwargs)
model.config.problem_type = "single_label_classification"

_eval_bs = 8 if "large" in MODEL_NAME.lower() else 16

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=save_steps,
    logging_steps=log_steps,
    save_total_limit=2,
    learning_rate=learning_rate,
    warmup_steps=warmup_steps,
    use_cpu=not (is_cuda or is_mps),
    max_grad_norm=1.0,
    per_device_train_batch_size=profile["per_device_train_batch_size"],
    gradient_accumulation_steps=profile["gradient_accumulation_steps"],
    per_device_eval_batch_size=_eval_bs,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    report_to="none",
    disable_tqdm=True,
    max_steps=smoke_steps if smoke_steps > 0 else -1,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokens,
    eval_dataset=dev_tokens,
    compute_metrics=compute_metrics,
    data_collator=StripLangCollator(tokenizer),
    class_weights=class_weights_tensor,
    label_smoothing=label_smoothing,
)

train_result = trainer.train()

training_log_csv = Path(CHECKPOINT_DIR) / "training_log_history.csv"
log_history = trainer.state.log_history
if log_history:
    log_keys = sorted({k for row in log_history for k in row})
    pd.DataFrame(log_history, columns=log_keys).to_csv(training_log_csv, index=False)

print(
    f"\nTraining complete: {train_result.global_step} optimizer steps, "
    f"final train_loss={train_result.training_loss:.3f}"
)
print(f"Wrote training log: {training_log_csv} ({len(log_history)} rows)")


## 5. Architecture comparison vs `Davlan/afro-xlmr-large-76L`

The AfriHate paper highlights `Davlan/afro-xlmr-large-76L` (a 24-layer, 1024-hidden encoder pretrained on 76 African languages) as a strong fine-tuned baseline. We compare **pretrained architectures** here — fine-tuning 76L on MPS would take many hours, but reporting its capacity gives the reader a clean point of reference for our `afro-xlmr-base` setup. Set `NLP_MODEL=Davlan/afro-xlmr-large-76L` and re-run from §2 to actually train it.


In [6]:
architecture_comparison_models = {"this_run": MODEL_NAME}
if MODEL_NAME != "Davlan/afro-xlmr-large-76L":
    architecture_comparison_models["paper_AfroXLMR_76L"] = "Davlan/afro-xlmr-large-76L"
if MODEL_NAME != "Davlan/afro-xlmr-base":
    architecture_comparison_models["smaller_AfroXLMR_base"] = "Davlan/afro-xlmr-base"

architecture_rows = []
for role, hub_id in architecture_comparison_models.items():
    try:
        cfg = AutoConfig.from_pretrained(hub_id, token=hf_token or None)
        architecture_rows.append({
            "role": role,
            "hub_model_id": hub_id,
            "hidden_size": cfg.hidden_size,
            "num_hidden_layers": cfg.num_hidden_layers,
            "num_attention_heads": cfg.num_attention_heads,
            "intermediate_size": cfg.intermediate_size,
            "vocab_size": cfg.vocab_size,
            "max_position_embeddings": cfg.max_position_embeddings,
        })
    except Exception as exc:
        architecture_rows.append({"role": role, "hub_model_id": hub_id, "error": str(exc)})

architecture_comparison_df = pd.DataFrame(architecture_rows)
print(architecture_comparison_df.to_string(index=False))


                 role               hub_model_id  hidden_size  num_hidden_layers  num_attention_heads  intermediate_size  vocab_size  max_position_embeddings
             this_run Davlan/afro-xlmr-large-76L         1024                 24                   16               4096      250002                      514
smaller_AfroXLMR_base      Davlan/afro-xlmr-base          768                 12                   12               3072      250002                      514


## 6. Zero-shot transfer — primary outcome

We apply the source-trained model directly to **Twi (`twi`)** and **Nigerian Pidgin (`pcm`)** AfriHate test sets — no fine-tuning, no target labels seen during training. Targets can be overridden via `NLP_ZERO_SHOT_LANGS` (comma-separated configs).

**What we expect:**

- If transfer is strong, zero-shot macro-F1 should be only modestly below the source-test ceiling (§7).
- If transfer is uneven across languages, the *ranking* of zero-shot F1 across targets will tell us which language families / typological neighbours benefit most from training on Hausa + Amharic.

The same metric panel as the dev / test evaluation is reported, so the consolidated report table (§8) compares **like-for-like**.


In [7]:
ZERO_SHOT_LANGS = [
    cfg.strip()
    for cfg in os.environ.get("NLP_ZERO_SHOT_LANGS", "twi,pcm").split(",")
    if cfg.strip()
]


def load_target_test(config: str):
    raw = load_dataset("afrihate/afrihate", config, token=hf_token)["test"]
    raw = raw.add_column("lang", [config] * len(raw))
    out_of_schema = set(raw["label"]) - set(label2id.keys())
    if out_of_schema:
        print(f"[{config}] dropping rows with labels outside source schema: {out_of_schema}")
    raw = raw.filter(lambda ex: ex["label"] in label2id)
    raw = raw.map(encode_labels).remove_columns(["label"])
    return tokenize_dataset(raw)


zero_shot_rows = []
for target_lang in ZERO_SHOT_LANGS:
    try:
        target_tokens = load_target_test(target_lang)
        if len(target_tokens) == 0:
            zero_shot_rows.append({"target": target_lang, "n": 0, "note": "no rows after alignment"})
            continue
        prediction = trainer.predict(target_tokens)
        target_metrics = evaluate_predictions(
            prediction.label_ids,
            np.argmax(prediction.predictions.astype(np.float64), axis=-1),
        )
        zero_shot_rows.append({"target": target_lang, "n": int(len(target_tokens)), **target_metrics})
    except Exception as exc:
        zero_shot_rows.append({"target": target_lang, "n": None, "error": str(exc)})

zero_shot_df = pd.DataFrame(zero_shot_rows)
print("=== Zero-shot transfer (no parameter updates on targets) ===\n")
print(zero_shot_df.to_string(index=False))


=== Zero-shot transfer (no parameter updates on targets) ===

target    n  accuracy  balanced_accuracy      mcc  f1_macro  f1_weighted  precision_macro  recall_macro  precision_weighted  recall_weighted  pred_majority_frac  num_pred_classes_used  f1_Abuse  f1_Hate  f1_Normal
   twi  698  0.527221           0.402083 0.093448  0.375106     0.543616         0.381141      0.402083            0.580236         0.527221            0.595989                    3.0  0.676211 0.142857   0.306250
   pcm 1593  0.608286           0.620124 0.383369  0.584453     0.593645         0.590239      0.620124            0.631557         0.608286            0.564972                    3.0  0.524980 0.531646   0.696732


## 7. In-distribution sanity check (Hausa + Amharic test)

If §6 zero-shot scores are weak but here the model is strong, we have a transfer problem (the research question). If both are weak, training itself is broken and zero-shot numbers cannot be interpreted. We also break down by source language (Hausa vs Amharic) to check that one isn't carrying the other.


In [8]:
print("Predicting on official Hausa + Amharic test split...")
test_prediction = trainer.predict(test_tokens)
test_predictions = np.argmax(test_prediction.predictions.astype(np.float64), axis=-1)
test_labels = np.asarray(test_prediction.label_ids).astype(np.int64).ravel()
test_metrics = evaluate_predictions(test_labels, test_predictions)

print("\nIn-distribution metrics (Hausa + Amharic):")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

true_label_counts = dict(zip(label_names, np.bincount(test_labels, minlength=len(label_names)).tolist()))
pred_label_counts = dict(zip(label_names, np.bincount(test_predictions, minlength=len(label_names)).tolist()))
print(f"\nTrue-label counts: {true_label_counts}")
print(f"Pred-label counts: {pred_label_counts}")

print("\nConfusion matrix (rows=true, cols=pred):")
print(confusion_matrix(test_labels, test_predictions, labels=list(range(len(label_names)))))

print("\nPer-class report:")
print(
    classification_report(
        test_labels,
        test_predictions,
        labels=list(range(len(label_names))),
        target_names=label_names,
        zero_division=0,
    )
)

if "lang" in test_tokens.column_names:
    test_langs = np.array(test_tokens["lang"])
    print("Per-source-language scores:")
    for source_lang in sorted(set(test_langs.tolist())):
        mask = test_langs == source_lang
        if not np.any(mask):
            continue
        per_lang = evaluate_predictions(test_labels[mask], test_predictions[mask])
        print(
            f"  [{source_lang}] n={int(mask.sum())} "
            f"acc={per_lang['accuracy']:.4f} "
            f"bacc={per_lang['balanced_accuracy']:.4f} "
            f"f1_macro={per_lang['f1_macro']:.4f} "
            f"mcc={per_lang['mcc']:.4f}"
        )

trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)
print(f"\nSaved final model and tokenizer to {FINAL_MODEL_PATH}")


Predicting on official Hausa + Amharic test split...



In-distribution metrics (Hausa + Amharic):
  accuracy: 0.7709
  balanced_accuracy: 0.7462
  mcc: 0.6409
  f1_macro: 0.7527
  f1_weighted: 0.7689
  precision_macro: 0.7632
  recall_macro: 0.7462
  precision_weighted: 0.7698
  recall_weighted: 0.7709
  pred_majority_frac: 0.4367
  num_pred_classes_used: 3.0000
  f1_Abuse: 0.7752
  f1_Hate: 0.6706
  f1_Normal: 0.8123

True-label counts: {'Abuse': 954, 'Hate': 550, 'Normal': 1111}
Pred-label counts: {'Abuse': 1012, 'Hate': 461, 'Normal': 1142}

Confusion matrix (rows=true, cols=pred):
[[762  67 125]
 [109 339 102]
 [141  55 915]]

Per-class report:
              precision    recall  f1-score   support

       Abuse       0.75      0.80      0.78       954
        Hate       0.74      0.62      0.67       550
      Normal       0.80      0.82      0.81      1111

    accuracy                           0.77      2615
   macro avg       0.76      0.75      0.75      2615
weighted avg       0.77      0.77      0.77      2615

Per-source-langu


Saved final model and tokenizer to /Users/macpro/Desktop/MastersICS/Natural Language Processing/Final_Project/FinalProject/Final_Source_Model


## 8. Consolidated report table

One CSV that carries the **same metric panel** for every evaluation setting — zero-shot targets and the in-distribution source test — so the write-up can plot or cite without re-deriving anything.


In [ ]:
report_columns = [
    "setting",
    "subset",
    "n_eval",
    "accuracy",
    "balanced_accuracy",
    "mcc",
    "f1_macro",
    "f1_weighted",
    "precision_macro",
    "recall_macro",
    "pred_majority_frac",
    "num_pred_classes_used",
    "notes",
]


def make_row(setting: str, subset: str, n_eval, metrics: dict | None, note: str = "") -> dict:
    base = {col: None for col in report_columns}
    base.update({"setting": setting, "subset": subset, "n_eval": n_eval, "notes": note})
    if metrics:
        for col in (
            "accuracy",
            "balanced_accuracy",
            "mcc",
            "f1_macro",
            "f1_weighted",
            "precision_macro",
            "recall_macro",
            "pred_majority_frac",
            "num_pred_classes_used",
        ):
            base[col] = metrics.get(col)
    return base


report_rows = []
for _, row in zero_shot_df.iterrows():
    report_rows.append(make_row(
        setting="zero_shot",
        subset=row.get("target"),
        n_eval=row.get("n"),
        metrics=row.to_dict(),
        note=row.get("error") if "error" in row and pd.notna(row.get("error")) else (row.get("note") or ""),
    ))

report_rows.append(make_row(
    setting="source_test_in_distribution",
    subset=SOURCE_TEST_SUBSET_LABEL,
    n_eval=int(len(test_tokens)),
    metrics=test_metrics,
))

report_table = pd.DataFrame(report_rows, columns=report_columns)
print("=== This run's report table ===\n")
print(report_table.to_string(index=False))

report_csv = Path(CHECKPOINT_DIR) / "results_report_table.csv"
report_table.to_csv(report_csv, index=False)
print(f"\nSaved (this run only): {report_csv}")

experiment_log_path = Path(CHECKPOINT_DIR) / "experiment_log.csv"
log_columns = [
    "experiment_id",
    "model",
    "source_languages",
    "num_epochs",
    "learning_rate",
    *report_columns,
]
log_rows = []
for row in report_rows:
    log_rows.append({
        "experiment_id": EXPERIMENT_ID,
        "model": MODEL_NAME,
        "source_languages": "+".join(SOURCE_LANGS),
        "num_epochs": num_epochs,
        "learning_rate": learning_rate,
        **row,
    })
log_df_new = pd.DataFrame(log_rows, columns=log_columns)
if experiment_log_path.exists():
    previous_log = pd.read_csv(experiment_log_path)
    keep_mask = ~(
        (previous_log["experiment_id"] == EXPERIMENT_ID)
        & (previous_log["model"] == MODEL_NAME)
        & (previous_log["source_languages"] == "+".join(SOURCE_LANGS))
    )
    combined_log = pd.concat([previous_log[keep_mask], log_df_new], ignore_index=True)
else:
    combined_log = log_df_new
combined_log.to_csv(experiment_log_path, index=False)

print(f"\n=== Aggregated experiment log ({experiment_log_path}) ===\n")
display_cols = [
    "experiment_id",
    "model",
    "source_languages",
    "setting",
    "subset",
    "f1_macro",
    "balanced_accuracy",
    "mcc",
    "pred_majority_frac",
]
print(combined_log[display_cols].to_string(index=False))


## 9. Findings, limitations, and next experiments

**Headline results (see `experiment_log.csv`).**

- **E1** (`afro-xlmr-base`, Hausa+Amharic): strong in-distribution ceiling; Twi zero-shot macro-F1 stays low (~0.29).
- **E2** (`afro-xlmr-large-76L`, Hausa+Amharic): higher source macro-F1 (~0.77) and better Pidgin zero-shot (~0.56) than E1; Twi macro-F1 can dip slightly vs E1 on the same split (transfer is not monotonic in model size alone).
- **E3** (`afro-xlmr-base`, Hausa+Amharic+**Yoruba**): adding a related Niger–Congo source lifts **Twi** zero-shot accuracy and macro-F1 substantially (~0.37 macro-F1) with little extra compute — our main *linguistic coverage* finding.
- **E4** (`afro-xlmr-large-76L`, Hausa+Amharic+Yoruba): best overall trade-off in our runs — **Pidgin** zero-shot macro-F1 **0.58**, **Twi** **0.38**, in-distribution macro-F1 **0.75** on the combined official test (2615 rows; 4-epoch run). Recommended for the write-up when MPS/VRAM allows.

**Limitations.**

- AfriHate label boundaries differ across languages; we drop out-of-schema target labels before evaluation.
- Zero-shot aggregate metrics hide per-class calibration; Phase 2 surfaces qualitative failure modes.

**Ethics.**

- Cross-lingual hate-speech models can over-flag in-group slang. High F1 does not imply deployment readiness — keep humans in the loop.

**Next steps.**

- Continue in `Phase2_FewShot_And_ErrorAnalysis.ipynb`: few-shot curves and error samples on the saved `Final_Source_Model/` checkpoint.
